In [ ]:
import pandas as pd
import os

def read_and_convert_to_mt_bert_df(csv_file:str):
    df = pd.read_csv(csv_file)
    df = df[["repository", "text", "label"]] # to single line
    df.rename(columns={"repository": "projectname", "text": "Abstract", "label": "original_label"}, inplace=True)
    return df

maldonado_train_comment = open('../config/baseline/origin/data--train.txt', 'r').readlines()
maldonado_train_label = open('../config/baseline/origin/label--train.txt', 'r').readlines()
maldonado_train_comment = list(map(lambda x: x.strip(), maldonado_train_comment))
maldonado_train_label = list(map(lambda x: x.strip(), maldonado_train_label))
maldonado_train_label = list(map(lambda x : "yes" if x.lower() == 'positive' else "no", maldonado_train_label))
guo_df = pd.DataFrame(data = {
    "projectname": ["all_projects"]* len(maldonado_train_comment),
    "Abstract": maldonado_train_comment,
    "original_label": maldonado_train_label,
    "datasetname": ["guo"]* len(maldonado_train_comment),
})
MT_BERT_FILE_FORMAT = "../cache/baseline/mt_bert/satd/multi_train/{}_data/{}.csv"
for ds in ["guo_duplicate", "guo_unique", "our_duplicate", "our_unique"]:
    train_file = MT_BERT_FILE_FORMAT.format(ds, f"{ds}_code_comments_train")
    test_file = MT_BERT_FILE_FORMAT.format(ds, f"{ds}_code_comments_test")
    for file in [train_file, test_file]:
        os.makedirs(os.path.dirname(file), exist_ok=True)

    if "guo" in ds:
        guo_df.to_csv(train_file, index=False)

    if "duplicate" in ds:
        if "guo" not in ds:
            read_and_convert_to_mt_bert_df("../data/duplicate_detect_train.csv").to_csv(train_file, index=False)
        read_and_convert_to_mt_bert_df("../data/duplicate_detect_test.csv").to_csv(test_file, index=False)
    if "unique" in ds:
        if "guo" not in ds:
            read_and_convert_to_mt_bert_df("../data/unique_detect_train.csv").to_csv(train_file, index=False)
        read_and_convert_to_mt_bert_df("../data/unique_detect_test.csv").to_csv(test_file, index=False)
    test_df = pd.read_csv(file)
    for dataset_type in ["train", "test"]:
        for source in ["commit", "issue", "pr"]:
            other_file = MT_BERT_FILE_FORMAT.format(ds, f"{ds}_{source}_{dataset_type}")
            os.makedirs(os.path.dirname(other_file), exist_ok=True)
            test_df[:10].to_csv(other_file, index=False)




In [ ]:
import pandas as pd
from Model import Model
from constant import *
from SimpleOutputLabelConverter import SimpleOutputLabelConverter
simple_output_label_converter = SimpleOutputLabelConverter({'yes', 'no'}, DEFAULT_DETECTION_CLASS)

detect_duplicate_test_df = pd.read_csv(f'../data/duplicate_detect_test.csv')
detect_duplicate_test_dataset = Dataset.from_pandas(detect_duplicate_test_df)

detect_unique_test_df = pd.read_csv(f'../data/unique_detect_test.csv')
detect_unique_test_dataset = Dataset.from_pandas(detect_unique_test_df)

for exp in ["guo_duplicate", "guo_unique", "our_duplicate", "our_unique"]:
    dataset_name = "duplicate" if "duplicate" in exp else "unique"
    dataset = detect_duplicate_test_dataset if "duplicate" == dataset_name else detect_unique_test_dataset
    pattern_model = Model('detect', f'bert-{exp}', simple_output_label_converter, 10_000)
    # pattern_model.fit(detect_train_dataset)
    output_file = "../cache/baseline/mt_bert/model/{}/results_code_comments_8.txt".format(exp)
    raw_predicted_labels = open(output_file, 'r').readlines()
    raw_predicted_labels = list(map(lambda x: x.strip(), raw_predicted_labels))
    predicted_labels = list(map(lambda x: "yes" if x == "1" else "no", raw_predicted_labels))
    pattern_model.predict_end(dataset, dataset_name, predicted_labels, raw_predicted_labels)
